# 第5回 知識表現と推論

リンクと規則を手で追ってから、同じ結果になるかを確かめます。標準ライブラリのみを使い、このNotebook単体で実行できます。

1. Pythonのカーネルを選びます。
2. カーネルを再起動します。
3. 上から全セルを実行します。

図や文章の参照がなくても演習ができるよう、入力と規則を下に収録しています。

## 入力と予想

端末の分類は昨年度isa.pyに基づく教材内の定義です。現実の機器分類を網羅しません。スマホ→携帯型端末→コンピュータをたどると「はい」。経路がない場合は不明とします。

架空の貸出規則は、R1：登録済み＆講習修了→利用資格あり、R2：利用資格あり＆在庫あり→貸出候補、R3：貸出候補＆予約確認済み→貸出準備可です。一件の申請のみを扱います。学校の実際の規則ではありません。

基準入力は登録済み・講習修了・在庫あり。R1、R2が適用され、予約確認がないのでR3は適用されないはずです。外部データ・乱数は使いません。

In [ ]:
# 昨年度isa.pyの分類を教材内の定義として使用。現実の機器分類を網羅しない。
PARENTS = {
    "携帯型端末": "コンピュータ", "ノートPC": "コンピュータ",
    "据え置き型PC": "コンピュータ", "スマホ": "携帯型端末",
    "タブレット": "携帯型端末", "デスクトップ": "据え置き型PC",
    "サーバ": "据え置き型PC",
}
# 架空の機器貸出。一件の申請だけを扱い、実際の学校の規則ではない。
FACTS = ("利用登録済み", "講習修了", "在庫あり")
RULES = (
    ("R1", ("利用登録済み", "講習修了"), "利用資格あり"),
    ("R2", ("利用資格あり", "在庫あり"), "貸出候補"),
    ("R3", ("貸出候補", "予約確認済み"), "貸出準備可"),
)


def isa_path(parents, start, target):
    """単一親の経路を返す。未到達は否定せずunknown、循環はデータの問題。"""
    known = set(parents) | set(parents.values())
    if start not in known or target not in known:
        return {"status": "unknown", "reason": "unknown_term", "path": []}
    path = [start]
    visited = set()
    current = start
    while True:
        if current == target:
            return {"status": "yes", "reason": "reachable", "path": path}
        if current in visited:
            return {"status": "cycle", "reason": "repeated_node", "path": path}
        visited.add(current)
        if current not in parents:
            return {"status": "unknown", "reason": "no_path", "path": path}
        current = parents[current]
        path.append(current)


def forward_chain(initial_facts, rules):
    """肯定的な事実を追加する最小推論器。規則順に照合し、追加時だけ記録。"""
    facts = set(initial_facts)  # 呼び出し元の事実は変えない
    trace = []
    changed = True
    while changed:
        changed = False
        for rule_id, conditions, conclusion in rules:
            if set(conditions) <= facts and conclusion not in facts:
                facts.add(conclusion)
                trace.append({"rule": rule_id, "conditions": list(conditions),
                              "conclusion": conclusion})
                changed = True
    return {"facts": sorted(facts), "trace": trace}


def demo():
    queries = (("スマホ", "コンピュータ"), ("サーバ", "コンピュータ"),
               ("スマホ", "ノートPC"), ("ENIAC", "コンピュータ"),
               ("スマホ", "電子機器"), ("コンピュータ", "コンピュータ"))
    return {
        "isa": [{"start": a, "target": b, **isa_path(PARENTS, a, b)} for a, b in queries],
        "cycle": isa_path({"A": "B", "B": "A", "C": "D"}, "A", "D"),
        "initial_facts": list(FACTS),
        "baseline": forward_chain(FACTS, RULES),
        "with_reservation": forward_chain((*FACTS, "予約確認済み"), RULES),
        "without_training": forward_chain(tuple(f for f in FACTS if f != "講習修了"), RULES),
    }


## 基準と固定比較を実行

経路、最終事実、適用した規則を順に読みます。`baseline_result`は基準の照合用に残し、後の実験で上書きしません。

In [ ]:
import json
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 出力の読み方

`status`のyesは確認できた経路、unknownは未知語または経路なし、cycleは未到達のまま同じノードに戻った状態です。既知の同一概念は長さ0の経路で確認済みとします。未知語同士の同一名は不明です。

`baseline`ではR1→R2、`with_reservation`ではR1→R2→R3、`without_training`では適用規則なしになります。factsは比較しやすい文字列順で、推論の順序ではありません。推論の順序はtraceを読みます。

循環例A→B→Aは有限回で止まります。循環より先に目標へ到達した場合は経路ありです。知識全体の無循環性を検査する機能ではありません。

## 自分で一つ変える：予約確認を追加

まず結論を予想し、次のセルを実行します。初期事実のコピーに一つ追加するので、元のFACTSは変わりません。

In [ ]:
trial_facts = set(FACTS)
trial_facts.add("予約確認済み")
trial_result = forward_chain(trial_facts, RULES)
print(json.dumps(trial_result, ensure_ascii=False, indent=2))
print("元の事実:", FACTS)


## 自分で一つ変える：知識を追加

電子機器は初めは未登録です。コピーに「コンピュータ→電子機器」を追加してから、同じ問いを試します。この関係も教材内で与える定義です。

In [ ]:
trial_parents = dict(PARENTS)
print("追加前:", isa_path(PARENTS, "スマホ", "電子機器"))
trial_parents["コンピュータ"] = "電子機器"
print("追加後:", isa_path(trial_parents, "スマホ", "電子機器"))
print("元の知識は保持:", "コンピュータ" not in PARENTS)


## 振り返り

- どのリンク、またはどの条件が結果を変えましたか。
- 導けない結論を、なぜ否定と断定しないのでしょうか。
- R3、R2、R1の順に規則を並べたら、一巡で終わるでしょうか。

必要なら別セルでコピーを作って確かめます。後ろ向き推論・フレームの既定値・明示的否定・複数申請・RETEはこのプログラムには実装していません。